In [0]:
DROP TABLE IF EXISTS workspace.default.orders_source;
DROP TABLE IF EXISTS workspace.default.orders_regular;
DROP TABLE IF EXISTS workspace.default.orders_partitioned;
DROP TABLE IF EXISTS workspace.default.orders_zorder;
DROP TABLE IF EXISTS workspace.default.orders_liquid;

Part1 - Source Preparation

In [0]:
--Create the source table
CREATE OR REPLACE TABLE workspace.default.orders_source
USING DELTA
AS
SELECT 
    CAST(id AS INT) AS order_id,
    CAST(pmod(id, 1000)+1 AS INT) AS customer_id,

    DATE_ADD(
        DATE '2025-01-01',
        CAST(pmod(id,365) AS INT)
    ) AS order_date,

    date_format(
        DATE_ADD (
            DATE '2025-01-01',
            CAST(pmod(id,365) AS INT)
        ),
        'yyyy-MM'
    ) AS order_month,
    CASE
        WHEN pmod(id, 4) = 0 THEN 'London'
        WHEN pmod(id, 4) = 1 THEN 'Coventry'
        WHEN pmod(id, 4) = 2 THEN 'Bristol'
        ELSE 'Manchester'
    END AS region,
    CAST(pmod(id*37,100000)/100.0 AS DOUBLE) AS amount
FROM range(10000000);


In [0]:
--validate source data
SELECT * FROM workspace.default.orders_source ;

In [0]:
SELECT 
    COUNT(*) AS total_orders,
    COUNT(DISTINCT customer_id) AS total_customers,
    COUNT(DISTINCT region) AS total_regions,
    MIN(order_date) AS min_order_date,
    MAX(order_date) AS max_order_date,
    SUM(amount) AS total_amount
FROM workspace.default.orders_source;

##### Part2 - Understanding traditional partitioning

In [0]:
-- Create a partitioned Delta Table
CREATE OR REPLACE TABLE workspace.default.orders_partitioned
USING DELTA
PARTITIONED BY (order_month)
AS
SELECT * FROM workspace.default.orders_source;


In [0]:
--Query partitioned columns
SELECT 
    order_month, 
    COUNT(*) AS total_orders,
    round(SUM(amount),2) AS total_sales
 FROM workspace.default.orders_partitioned 
 WHERE order_month = '2025-03'
 GROUP BY order_month;


####Inspect partitions metadata

In [0]:
DESCRIBE DETAIL workspace.default.orders_partitioned;

#### Part3 - Understand Z-ORDER

In [0]:
--Create a regular Delta table for Z-order
CREATE OR REPLACE TABLE workspace.default.orders_zorder
USING DELTA
AS
SELECT * FROM workspace.default.orders_source;

##### Apply Z-ORDER

In [0]:
OPTIMIZE workspace.default.orders_zorder ZORDER BY (customer_id, order_month);


##### Query the Z-ORDER table

In [0]:
SELECT * FROM workspace.default.orders_zorder WHERE customer_id = 500;

#### Part 4 - Liquid Clustering hands-on

In [0]:
--Create a liquid clustering table
CREATE OR REPLACE TABLE workspace.default.orders_liquid
USING DELTA
CLUSTER BY (customer_id,order_date)
AS
SELECT * FROM workspace.default.orders_source;

##### Inspect table structre

In [0]:
DESCRIBE TABLE workspace.default.orders_liquid;

#### Inspect Liquid Clustering Metadata

In [0]:
DESCRIBE DETAIL workspace.default.orders_liquid;

#### Run OPTIMIZE

In [0]:
OPTIMIZE workspace.default.orders_liquid;

##### Query using the first clsutering column

In [0]:
SELECT * FROM workspace.default.orders_liquid WHERE customer_id = 500;

#### Part 5 -- Change Liquid Clustering Keys

In [0]:
ALTER TABLE workspace.default.orders_liquid CLUSTER BY (region,order_date);

Recluster existing data

In [0]:
OPTIMIZE workspace.default.orders_liquid FULL;

In [0]:
DESCRIBE DETAIL workspace.default.orders_liquid;
